## Pipeline di caricamento dati su SQL
In questo notebook configuriamo una pipeline per il caricamento massivo dei dati nel database MariaDB. Iniziamo importando le librerie necessarie per la manipolazione dei dati (`pandas`) e per la gestione della connessione al database (`sqlalchemy`).



In [38]:
import pandas as pd
from sqlalchemy import create_engine

### Caricamento del Dataset Pulito
Ricarichiamo il file CSV precedentemente pulito e processato nella fase di EDA. Questo file contiene tutte le informazioni necessarie, ma è ancora in formato "flat" (piatto), privo degli identificativi numerici necessari per lo schema relazionale SQL.

In [39]:
url= r"C:\Users\angel\OneDrive\Desktop\Lavoro STAFF\hotel_reviews_clean.csv"
hr1= pd.read_csv(url)

### Connessione al Database e recupero delle Primary Keys
Stabiliamo la connessione con il database locale tramite l'engine di SQLAlchemy. Recuperiamo le tabelle `hotels` e `reviewers` per ottenere gli ID univoci (Primary Keys) generati da SQL. Questi ID sono fondamentali per popolare correttamente la tabella delle recensioni rispettando i vincoli di integrità referenziale.


In [40]:
engine = create_engine('mysql+mysqlconnector://root:@localhost/Hotel_Reviews')

sql_hotels = pd.read_sql("SELECT hotel_id, hotel_name FROM hotels", engine)
sql_reviewers = pd.read_sql("SELECT reviewer_id, reviewer_nationality FROM reviewers", engine)


### Mapping dei dati (Data Join)
Eseguiamo un'operazione di `merge` (equivalente alla JOIN di SQL) tra il dataset originale e le tabelle delle anagrafiche appena scaricate. Questo passaggio ci permette di associare a ogni recensione l'ID corrispondente dell'hotel e del recensore, trasformando i nomi testuali in riferimenti numerici.


In [41]:
hr_final = hr1.merge(sql_hotels, left_on='Hotel_Name', right_on='hotel_name')
hr_final = hr_final.merge(sql_reviewers, left_on='Reviewer_Nationality', right_on='reviewer_nationality')

### Preparazione finale e Caricamento Massivo (Bulk Insert)
Selezioniamo solo le colonne richieste dalla tabella di destinazione e rinominiamo i campi per farli combaciare perfettamente con lo schema SQL. 
Infine, utilizziamo il metodo `to_sql` con un parametro di `chunksize` per inviare i dati a blocchi di 10.000 righe. Questa tecnica di "chunking" previene il timeout del server e garantisce un caricamento fluido di oltre 500.000 record.


In [42]:
final_reviews = hr_final[['hotel_id', 'reviewer_id', 'Review_Date', 'Reviewer_Score', 'Positive_Review', 'Negative_Review']]

final_reviews.columns = ['hotel_id', 'reviewer_id', 'review_date', 'reviewer_score', 'positive_review', 'negative_review']

print(" Pipeline completata! Gli ID sono stati collegati e i dati caricati.")


 Pipeline completata! Gli ID sono stati collegati e i dati caricati.


In [43]:
final_reviews.to_sql(
    name='reviews', 
    con=engine, 
    if_exists='append', 
    index=False, 
    chunksize=10000
)

print( "Operazione completata! Le 515.000 recensioni sono ora su phpMyAdmin.")


Operazione completata! Le 515.000 recensioni sono ora su phpMyAdmin.
